In [5]:
import os
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms
from PIL import Image

In [6]:
class imageProcessor:
    def __init__(self, root_dir_path, transformations=None):
        self.root_dir_path = root_dir_path
        self.transformations = transformations

        image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

        self.all_img_paths = [
            os.path.join(root_dir_path, img)
            for img in os.listdir(root_dir_path)
            if img.lower().endswith(image_extensions)
        ]

    def __len__(self):
        return len(self.all_img_paths)

    def __getitem__(self, idx):
        img_path = self.all_img_paths[idx]

        img = Image.open(img_path).convert("RGB")

        if self.transformations:
            img = self.transformations(img)

        return img

In [9]:
root_dir_path = "./img_align_celeba"

transformations = transforms.Compose(
    [
        transforms.CenterCrop(178),
        transforms.Resize(64),
        transforms.ToTensor(),
        transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
    ]
)

In [10]:
Dataset = imageProcessor(root_dir_path,transformations)
print(f"loaded images:{len(Dataset)}")

loaded images:202599


In [11]:
import os

print("Current directory:")
print(os.getcwd())

print("\nFiles/Folders here:")
print(os.listdir())

Current directory:
c:\Users\jinju\Documents\AI_ML\Prime_\Deep_Learing\GANs

Files/Folders here:
['DCGAN.ipynb', 'img_align_celeba', 'vanila_GANs.ipynb']


In [ ]:
#%matplotlib inline
import argparse
import os
import random
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.utils.data
import torchvision.transforms as transforms
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt




Inputs
======

Let's define some inputs for the run:

-   `dataroot` - the path to the root of the dataset folder. We will
    talk more about the dataset in the next section.
-   `workers` - the number of worker threads for loading the data with
    the `DataLoader`.
-   `batch_size` - the batch size used in training. The DCGAN paper uses
    a batch size of 128.
-   `image_size` - the spatial size of the images used for training.
    This implementation defaults to 64x64. If another size is desired,
    the structures of D and G must be changed. See
    [here](https://github.com/pytorch/examples/issues/70) for more
    details.
-   `nc` - number of color channels in the input images. For color
    images this is 3.
-   `nz` - length of latent vector.
-   `ngf` - relates to the depth of feature maps carried through the
    generator.
-   `ndf` - sets the depth of feature maps propagated through the
    discriminator.
-   `num_epochs` - number of training epochs to run. Training for longer
    will probably lead to better results but will also take much longer.
-   `lr` - learning rate for training. As described in the DCGAN paper,
    this number should be 0.0002.
-   `beta1` - beta1 hyperparameter for Adam optimizers. As described in
    paper, this number should be 0.5.
-   `ngpu` - number of GPUs available. If this is 0, code will run in
    CPU mode. If this number is greater than 0 it will run on that
    number of GPUs.
